# BirdCLEF 2026 - ProtoSSM v5

This notebook implements the Maximum Ensemble pipeline using a Selective State Space Model (ProtoSSM v5) enhanced with a taxonomic auxiliary head, cross-attention, and a Residual SSM for second-pass boosting.

## 1. Setup & Environment Configuration

We consolidate all hyperparameters into a single robust configuration class. This guarantees a single source of truth for the entire pipeline and avoids scattered dictionary updates.

In [1]:
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorboard-2.20.0-py3-none-any.whl
!pip install -q --no-deps /kaggle/input/notebooks/ashok205/tf-wheels/tf_wheels/tensorflow-2.20.0-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl

In [2]:
import os
import gc
import json
import re
import time
import warnings
from collections import defaultdict
from pathlib import Path
from typing import Dict, List, Tuple, Optional, Any, Union

import numpy as np
import pandas as pd
import soundfile as sf
from tqdm.auto import tqdm

import tensorflow as tf
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.isotonic import IsotonicRegression

warnings.filterwarnings("ignore")
tf.experimental.numpy.experimental_enable_numpy_behavior()

# Environment constraints for Kaggle
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = ""

class Config:
    """Centralized configuration for the ProtoSSM v5 Maximum Ensemble."""
    
    # Execution Mode
    MODE: str = "submit"  # 'train' or 'submit'
    VERBOSE: bool = (MODE == "train")
    
    # Constants
    SR: int = 32000
    WINDOW_SEC: int = 5
    WINDOW_SAMPLES: int = SR * WINDOW_SEC
    FILE_SAMPLES: int = 60 * SR
    N_WINDOWS: int = 12
    DEVICE: torch.device = torch.device("cpu")
    
    # Paths
    BASE_DIR: Path = Path("/kaggle/input/competitions/birdclef-2026")
    MODEL_DIR: Path = Path("/kaggle/input/models/google/bird-vocalization-classifier/tensorflow2/perch_v2_cpu/1")
    FULL_CACHE_INPUT_DIR: Path = Path("/kaggle/input/datasets/jaejohn/perch-meta")
    FULL_CACHE_WORK_DIR: Path = Path("/kaggle/working/perch_cache")
    
    # Inference Cache & Batching
    BATCH_FILES: int = 16
    PROXY_REDUCE: str = "max"
    REQUIRE_FULL_CACHE_IN_SUBMIT: bool = False
    DRYRUN_N_FILES: int = 50 if MODE == "train" else 20
    
    # ProtoSSM v5 Architecture
    PROTO_SSM: Dict[str, Any] = {
        "d_model": 320,
        "d_state": 32,
        "n_ssm_layers": 4,
        "dropout": 0.12,
        "n_prototypes": 2,
        "n_sites": 20,
        "meta_dim": 24,
        "use_cross_attn": True,
        "cross_attn_heads": 8,
    }
    
    # ProtoSSM Training Parameters
    PROTO_SSM_TRAIN: Dict[str, Any] = {
        "n_epochs": 80,  # V18: unconditional (was 80 if train else 40)
        "lr": 8e-4,
        "weight_decay": 1e-3,
        "val_ratio": 0.15,
        "patience": 20,  # V18: unconditional (was 20 if train else 8)
        "pos_weight_cap": 25.0,
        "distill_weight": 0.15,
        "proto_margin": 0.15,
        "label_smoothing": 0.03,
        "oof_n_splits": 5,
        "mixup_alpha": 0.4,
        "focal_gamma": 2.5,
        "swa_start_frac": 0.65,
        "swa_lr": 4e-4,
        "use_cosine_restart": True,
        "restart_period": 20,
    }
    
    # Residual SSM
    RESIDUAL_SSM: Dict[str, Any] = {
        "d_model": 128,
        "d_state": 16,
        "n_ssm_layers": 2,
        "dropout": 0.1,
        "correction_weight": 0.35,
        "n_epochs": 40,
        "lr": 8e-4,
        "patience": 12,
    }
    
    # Post-Processing & Thresholds
    FILE_LEVEL_TOP_K: int = 2
    TTA_SHIFTS: List[int] = [0, 1, -1, 2, -2]
    RANK_AWARE_SCALE: bool = True
    RANK_AWARE_POWER: float = 0.4
    DELTA_SHIFT_ALPHA: float = 0.20
    THRESHOLD_GRID: List[float] = [0.25, 0.30, 0.35, 0.40, 0.45, 0.50, 0.55, 0.60, 0.65, 0.70]
    
    # Fusion Parameters
    BEST_FUSION: Dict[str, float] = {
        "lambda_event": 0.45,
        "lambda_texture": 1.1,
        "lambda_proxy_texture": 0.9,
        "smooth_texture": 0.35,
        "smooth_event": 0.15,
    }
    
    TEMPERATURE: Dict[str, float] = {
        "aves": 1.10,
        "texture": 0.95,
    }
    
    # Probe Modeling Parameters
    PROBE_BACKEND: str = "mlp"
    MLP_PARAMS: Dict[str, Any] = {
        "hidden_layer_sizes": (256, 128),
        "activation": "relu",
        "max_iter": 500,
        "early_stopping": True,
        "validation_fraction": 0.15,
        "n_iter_no_change": 20,
        "random_state": 42,
        "learning_rate_init": 5e-4,
        "alpha": 0.005,
    }
    
    FROZEN_BEST_PROBE: Dict[str, float] = {
        "pca_dim": 128,
        "min_pos": 5,
        "C": 0.75,
        "alpha": 0.45,
    }

# Create necessary directories
Config.FULL_CACHE_WORK_DIR.mkdir(parents=True, exist_ok=True)
_WALL_START = time.time()

## 2. Advanced Training Utilities & Augmentations

These modules encapsulate our domain-specific deep learning enhancements. 
* **Species Focal Loss**: Mitigates class imbalance by down-weighting well-classified examples. The implementation applies the formula $FL(p_t) = -\alpha_t (1 - p_t)^\gamma \log(p_t)$ alongside binary cross-entropy.
* **Mixup+CutMix Hybrid**: Regularizes training by interpolating temporal embeddings across different soundscapes.

In [3]:
class TrainingUtils:
    """Encapsulates advanced loss functions, schedulers, and augmentations."""

    @staticmethod
    def get_scheduler(optimizer: torch.optim.Optimizer, epochs: int, steps_per_epoch: int, max_lr: float):
        """Restored OneCycleLR for superconvergence."""
        return torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=max_lr, epochs=epochs, steps_per_epoch=steps_per_epoch, pct_start=0.1, anneal_strategy='cos')

    @staticmethod
    def mixup_files(emb: np.ndarray, logits: np.ndarray, labels: np.ndarray,
                    site_ids: np.ndarray, hours: np.ndarray,
                    families: Optional[np.ndarray], alpha: float = 0.4
                    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray, Optional[np.ndarray]]:
        """File-level Mixup augmentation for ProtoSSM training.
        Ensures lam >= 0.5 so the dominant sample always contributes more.
        Discrete metadata (site, hour) is kept from the dominant sample unchanged."""
        n = len(emb)
        if alpha <= 0 or n < 2:
            return emb, logits, labels, site_ids, hours, families

        lam = np.random.beta(alpha, alpha)
        lam = max(lam, 1.0 - lam)  # dominant sample always contributes more
        perm = np.random.permutation(n)

        emb_mix    = lam * emb    + (1 - lam) * emb[perm]
        logits_mix = lam * logits + (1 - lam) * logits[perm]
        labels_mix = lam * labels + (1 - lam) * labels[perm]
        families_mix = (lam * families + (1 - lam) * families[perm]
                        if families is not None else None)

        return emb_mix, logits_mix, labels_mix, site_ids, hours, families_mix

    @staticmethod
    def focal_bce_with_logits(logits: torch.Tensor, targets: torch.Tensor, 
                              gamma: float = 2.0, pos_weight: Optional[torch.Tensor] = None, 
                              reduction: str = "mean") -> torch.Tensor:
        """Focal loss for multi-label bioacoustic classification."""
        if pos_weight is not None:
            bce = F.binary_cross_entropy_with_logits(logits, targets, pos_weight=pos_weight, reduction="none")
        else:
            bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        
        p = torch.sigmoid(logits)
        pt = targets * p + (1 - targets) * (1 - p)
        focal_weight = (1 - pt) ** gamma
        loss = focal_weight * bce
        
        return loss.mean() if reduction == "mean" else loss

    @staticmethod
    def build_class_freq_weights(y_full: np.ndarray, cap: float = 10.0) -> torch.Tensor:
        """Derives inverse square-root frequency weights for species balance."""
        pos_count = y_full.sum(axis=0).astype(np.float32) + 1.0
        total = y_full.shape[0]
        freq = pos_count / total
        weights = 1.0 / (freq ** 0.5)
        weights = np.clip(weights, 1.0, cap)
        weights = weights / weights.mean()
        return torch.tensor(weights, dtype=torch.float32)


## 3. Post-Processing Engine

Contains deterministic rank-aware adjustments, temporal delta shifting, and OOF-optimized thresholding logic designed to refine logits at inference time.

In [4]:
class PostProcessor:
    """Manages temporal smoothing, threshold optimizations, and rank-aware prediction scaling."""

    @staticmethod
    def rank_aware_scaling(scores: np.ndarray, n_windows: int = 12, power: float = 0.5) -> np.ndarray:
        """Scales window predictions exponentially by the file's max prediction confidence."""
        N, C = scores.shape
        assert N % n_windows == 0
        n_files = N // n_windows
        
        view = scores.reshape(n_files, n_windows, C)
        file_max = view.max(axis=1, keepdims=True)
        scale = np.power(file_max, power)
        scaled = view * scale
        return scaled.reshape(N, C)

    @staticmethod
    def adaptive_delta_smooth(probs: np.ndarray, n_windows: int = 12, base_alpha: float = 0.20) -> np.ndarray:
        """Restored adaptive, confidence-based interior smoothing."""
        n_files = probs.shape[0] // n_windows
        result = probs.copy()
        view = result.reshape(n_files, n_windows, -1)
        p_view = probs.reshape(n_files, n_windows, -1)
        for i in range(1, n_windows - 1):
            conf = p_view[:, i, :].max(axis=-1, keepdims=True)
            a = base_alpha * (1.0 - conf)
            view[:, i, :] = (1.0 - a) * p_view[:, i, :] + a * ((p_view[:, i-1, :] + p_view[:, i+1, :]) / 2.0)
        return result.reshape(probs.shape)

    @staticmethod
    def file_level_confidence_scale(preds: np.ndarray, n_windows: int = 12, top_k: int = 2) -> np.ndarray:
        """Scales window predictions by the file's top-K mean confidence."""
        N, C = preds.shape
        assert N % n_windows == 0
        view = preds.reshape(-1, n_windows, C)
        sorted_view = np.sort(view, axis=1)
        top_k_mean = sorted_view[:, -top_k:, :].mean(axis=1, keepdims=True)
        scaled = view * top_k_mean
        return scaled.reshape(N, C)

    @staticmethod
    def optimize_per_class_thresholds(oof_scores: np.ndarray, y_true: np.ndarray, 
                                      thresholds: List[float]) -> Tuple[np.ndarray, np.ndarray]:
        """Grid searches the optimal F1 threshold per class based on OOF stackings."""
        n_classes = oof_scores.shape[1]
        best_thresholds = np.zeros(n_classes)
        best_scores = np.zeros(n_classes)
        
        for c in range(n_classes):
            y_c = y_true[:, c]
            scores_c = oof_scores[:, c]
            
            if y_c.sum() == 0:
                best_thresholds[c] = 0.5
                continue
                
            best_f1, best_t = 0.0, 0.5
            for t in thresholds:
                pred_c = (scores_c > t).astype(int)
                tp = ((pred_c == 1) & (y_c == 1)).sum()
                fp = ((pred_c == 1) & (y_c == 0)).sum()
                fn = ((pred_c == 0) & (y_c == 1)).sum()
                
                if tp + fp == 0 or tp + fn == 0:
                    continue
                    
                precision = tp / (tp + fp)
                recall = tp / (tp + fn)
                f1 = 2 * precision * recall / (precision + recall + 1e-8)
                
                if f1 > best_f1:
                    best_f1 = f1
                    best_t = t
            
            best_thresholds[c] = best_t
            best_scores[c] = best_f1
            
        return best_thresholds, best_scores

    @staticmethod
    def apply_per_class_thresholds(scores: np.ndarray, thresholds: np.ndarray) -> np.ndarray:
        """Applies thresholds as a piecewise scaling factor to sharpen predictions."""
        N, C = scores.shape
        scaled = np.copy(scores)
        for c in range(C):
            t = thresholds[c]
            mask_above = scores[:, c] > t
            scaled[mask_above, c] = 0.5 + 0.5 * (scores[mask_above, c] - t) / (1 - t + 1e-8)
            scaled[~mask_above, c] = 0.5 * scores[~mask_above, c] / (t + 1e-8)
        return np.clip(scaled, 0.0, 1.0)

## 4. Taxonomy & Metadata Manager

This class shields the core logic from raw data ingestion logic. It manages file parsing, the generation of surrogate labels (Genus Proxies) for unseen species, and establishes the exact mapping to the underlying Perch index architecture.

In [5]:
class TaxonomyManager:
    """Handles raw data loading, label matrices, taxonomy grouping, and proxy mappings."""
    
    def __init__(self, cfg: type(Config)):
        self.cfg = cfg
        self.taxonomy = pd.read_csv(self.cfg.BASE_DIR / "taxonomy.csv")
        self.sample_sub = pd.read_csv(self.cfg.BASE_DIR / "sample_submission.csv")
        self.soundscape_labels = pd.read_csv(self.cfg.BASE_DIR / "train_soundscapes_labels.csv")
        
        self.primary_labels = self.sample_sub.columns[1:].tolist()
        self.n_classes = len(self.primary_labels)
        self.label_to_idx = {c: i for i, c in enumerate(self.primary_labels)}
        
        self.taxonomy["primary_label"] = self.taxonomy["primary_label"].astype(str)
        self.soundscape_labels["primary_label"] = self.soundscape_labels["primary_label"].astype(str)
        
        # Taxonomic auxiliary variables
        self.n_families = 0
        self.class_to_family = []
        
        self.sc_clean = pd.DataFrame()
        self.y_sc = np.array([])
        self.y_full_truth = np.array([])
        self.full_files = []
        
        # Deep Index tracking for Prior Fusion
        self.indices = {}
        self.selected_proxy_pos_to_bc = {}

    def _parse_soundscape_filename(self, name: str) -> dict:
        m = re.match(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg", name)
        if not m:
            return {"site": None, "hour_utc": -1}
        return {"site": m.group(2), "hour_utc": int(m.group(4)[:2])}

    def build_label_matrices(self) -> None:
        def union_labels(series):
            return sorted(set(lbl for x in series for lbl in ([t.strip() for t in str(x).split(";") if t.strip()] if pd.notna(x) else [])))
            
        clean_df = self.soundscape_labels.groupby(["filename", "start", "end"])["primary_label"].apply(union_labels).reset_index(name="label_list")
        clean_df["end_sec"] = pd.to_timedelta(clean_df["end"]).dt.total_seconds().astype(int)
        clean_df["row_id"] = clean_df["filename"].str.replace(".ogg", "", regex=False) + "_" + clean_df["end_sec"].astype(str)
        
        meta = clean_df["filename"].apply(self._parse_soundscape_filename).apply(pd.Series)
        self.sc_clean = pd.concat([clean_df, meta], axis=1)
        
        windows_per_file = self.sc_clean.groupby("filename").size()
        self.full_files = sorted(windows_per_file[windows_per_file == self.cfg.N_WINDOWS].index.tolist())
        self.sc_clean["file_fully_labeled"] = self.sc_clean["filename"].isin(self.full_files)

        self.y_sc = np.zeros((len(self.sc_clean), self.n_classes), dtype=np.uint8)
        for i, labels in enumerate(self.sc_clean["label_list"]):
            idxs = [self.label_to_idx[lbl] for lbl in labels if lbl in self.label_to_idx]
            if idxs:
                self.y_sc[i, idxs] = 1

        full_truth = self.sc_clean[self.sc_clean["file_fully_labeled"]].sort_values(["filename", "end_sec"]).reset_index(drop=False)
        self.y_full_truth = self.y_sc[full_truth["index"].to_numpy()]

    def build_taxonomy_groups(self) -> None:
        """Builds groups for the auxiliary taxonomy head matching original fallback logic."""
        group_map = {label: "Unknown" for label in self.primary_labels}
        
        # Original fallback logic
        for col in ["family", "order", "class_name"]:
            if col in self.taxonomy.columns:
                group_map = self.taxonomy.set_index("primary_label")[col].to_dict()
                break
                
        groups = sorted(set(group_map.values()))
        grp_to_idx = {g: i for i, g in enumerate(groups)}
        
        self.n_families = len(groups)
        self.class_to_family = [
            grp_to_idx.get(group_map.get(label, "Unknown"), 0) 
            for label in self.primary_labels
        ]

    def map_perch_indices(self) -> dict:
        """Resolves exact mappings, synonyms, and genus proxies."""
        bc_labels = pd.read_csv(self.cfg.MODEL_DIR / "assets" / "labels.csv").reset_index().rename(columns={"index": "bc_index", "inat2024_fsd50k": "scientific_name"})
        no_label_index = len(bc_labels)
        
        # Synonym handling
        MANUAL_SCIENTIFIC_NAME_MAP = {}
        self.taxonomy["scientific_name_lookup"] = self.taxonomy["scientific_name"].replace(MANUAL_SCIENTIFIC_NAME_MAP)
        bc_lookup = bc_labels.rename(columns={"scientific_name": "scientific_name_lookup"})
        
        mapping = self.taxonomy.merge(bc_lookup[["scientific_name_lookup", "bc_index"]], on="scientific_name_lookup", how="left")
        mapping["bc_index"] = mapping["bc_index"].fillna(no_label_index).astype(int)
        
        bc_indices = np.array([int(mapping.set_index("primary_label")["bc_index"].loc[c]) for c in self.primary_labels], dtype=np.int32)
        mapped_mask = bc_indices != no_label_index
        
        # Detailed indices for Prior Fusion
        class_name_map = self.taxonomy.set_index("primary_label")["class_name"].to_dict()
        texture_taxa = {"Amphibia", "Insecta"}
        active_classes = [self.primary_labels[i] for i in np.where(self.y_sc.sum(axis=0) > 0)[0]]
        
        idx_active_texture = np.array([self.label_to_idx[c] for c in active_classes if class_name_map.get(c) in texture_taxa], dtype=np.int32)
        idx_active_event = np.array([self.label_to_idx[c] for c in active_classes if class_name_map.get(c) not in texture_taxa], dtype=np.int32)
        unmapped_pos = np.where(~mapped_mask)[0].astype(np.int32)
        
        # Proxy Logic
        unmapped_df = mapping[mapping["bc_index"] == no_label_index].copy()
        unmapped_non_sonotype = unmapped_df[~unmapped_df["primary_label"].astype(str).str.contains("son", na=False)]
        
        proxy_map = {}
        for _, row in unmapped_non_sonotype.iterrows():
            genus = str(row["scientific_name"]).split()[0]
            hits = bc_labels[bc_labels["scientific_name"].astype(str).str.match(rf"^{re.escape(genus)}\s", na=False)]
            if len(hits) > 0:
                proxy_map[row["primary_label"]] = hits["bc_index"].astype(int).tolist()
                
        selected_proxy_targets = sorted([t for t in proxy_map.keys() if class_name_map.get(t) in {"Amphibia", "Insecta", "Aves"}])
        selected_proxy_pos = np.array([self.label_to_idx[c] for c in selected_proxy_targets], dtype=np.int32)
        self.selected_proxy_pos_to_bc = {self.label_to_idx[t]: np.array(proxy_map[t], dtype=np.int32) for t in selected_proxy_targets}

        # Save indices for fusion later
        self.indices = {
            "mapped_active_event": idx_active_event[mapped_mask[idx_active_event]],
            "mapped_active_texture": idx_active_texture[mapped_mask[idx_active_texture]],
            "selected_proxy_active_texture": np.intersect1d(selected_proxy_pos, idx_active_texture),
            "selected_prioronly_active_texture": np.setdiff1d(idx_active_texture[~mapped_mask[idx_active_texture]], selected_proxy_pos),
            "selected_prioronly_active_event": np.setdiff1d(idx_active_event[~mapped_mask[idx_active_event]], selected_proxy_pos),
            "unmapped_inactive": np.array([i for i in unmapped_pos if self.primary_labels[i] not in active_classes], dtype=np.int32),
            "active_texture": idx_active_texture,
            "active_event": idx_active_event
        }

        return {"bc_indices": bc_indices, "mapped_mask": mapped_mask, "mapped_pos": np.where(mapped_mask)[0].astype(np.int32), "class_name_map": class_name_map}

tax_manager = TaxonomyManager(Config)
tax_manager.build_label_matrices()
tax_manager.build_taxonomy_groups()
perch_mappings = tax_manager.map_perch_indices()

## 5. Core Model Architectures

This section defines the neural architectures. The primary model, **ProtoSSM v5**, is designed specifically for bioacoustics. 
Traditional approaches treat 5-second audio windows independently. ProtoSSM models the sequential dependency of these windows using a bidirectional Selective State Space Model (SSM) and a Temporal Cross-Attention layer. This captures critical temporal patterns such as the onset of a dawn chorus or species-specific counter-singing.

Additionally, we define the **Residual SSM**, a lightweight second-pass model trained explicitly on the *errors* (residuals) of the first-pass ensemble to correct systematic biases (e.g., consistent under-prediction of a specific species at a certain site).

In [6]:
class SelectiveSSM(nn.Module):
    """
    Simplified Mamba-style selective state space model.
    Provides input-dependent (selective) discretization of continuous-time SSMs.
    Efficient for short sequences (e.g., T=12 windows) on CPU.
    """
    def __init__(self, d_model: int, d_state: int = 16, d_conv: int = 4):
        super().__init__()
        self.d_model = d_model
        self.d_state = d_state

        self.in_proj = nn.Linear(d_model, 2 * d_model, bias=False)
        self.conv1d = nn.Conv1d(
            d_model, d_model, d_conv,
            padding=d_conv - 1, groups=d_model
        )
        self.dt_proj = nn.Linear(d_model, d_model, bias=True)

        A = torch.arange(1, d_state + 1, dtype=torch.float32).unsqueeze(0).expand(d_model, -1)
        self.A_log = nn.Parameter(torch.log(A))
        self.D = nn.Parameter(torch.ones(d_model))
        self.B_proj = nn.Linear(d_model, d_state, bias=False)
        self.C_proj = nn.Linear(d_model, d_state, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B_size, T, D = x.shape
        xz = self.in_proj(x)
        x_ssm, z = xz.chunk(2, dim=-1)

        x_conv = self.conv1d(x_ssm.transpose(1, 2))[:, :, :T].transpose(1, 2)
        x_conv = F.silu(x_conv)

        dt = F.softplus(self.dt_proj(x_conv))
        A = -torch.exp(self.A_log)
        B = self.B_proj(x_conv)
        C = self.C_proj(x_conv)

        h = torch.zeros(B_size, D, self.d_state, device=x.device)
        ys = []
        for t in range(T):
            dt_t = dt[:, t, :]
            dA = torch.exp(A[None, :, :] * dt_t[:, :, None])
            dB = dt_t[:, :, None] * B[:, t, None, :]
            h = h * dA + x[:, t, :, None] * dB
            y_t = (h * C[:, t, None, :]).sum(-1)
            ys.append(y_t)

        y = torch.stack(ys, dim=1)
        return y + x * self.D[None, None, :]


class TemporalCrossAttention(nn.Module):
    """
    Multi-head cross-attention between temporal windows.
    Captures non-local patterns that sequential SSMs might miss.
    """
    def __init__(self, d_model: int, n_heads: int = 4, dropout: float = 0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.norm = nn.LayerNorm(d_model)
        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.Dropout(dropout),
        )
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x: torch.Tensor) -> torch.Tensor:
        residual = x
        x = self.norm(x)
        attn_out, _ = self.attn(x, x, x)
        x = residual + attn_out
        
        residual = x
        x = self.norm2(x)
        x = residual + self.ffn(x)
        return x


class ProtoSSMv5(nn.Module):
    """
    Prototypical State Space Model v5.
    Features:
    - Bidirectional SSMs for local temporal sequence modeling.
    - Temporal Cross-Attention for global context.
    - Metadata awareness (Site & Hour embeddings).
    - Prototypical contrastive head initialized from data means.
    - Gated fusion with underlying Perch base logits.
    """
    def __init__(self, d_input: int = 1536, d_model: int = 320, d_state: int = 32,
                 n_ssm_layers: int = 4, n_classes: int = 234, n_windows: int = 12,
                 dropout: float = 0.12, n_sites: int = 20, meta_dim: int = 24,
                 use_cross_attn: bool = True, cross_attn_heads: int = 8):
        super().__init__()
        self.d_model = d_model
        self.n_classes = n_classes
        self.n_windows = n_windows

        self.input_proj = nn.Sequential(
            nn.Linear(d_input, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)

        self.ssm_fwd = nn.ModuleList()
        self.ssm_bwd = nn.ModuleList()
        self.ssm_merge = nn.ModuleList()
        self.ssm_norm = nn.ModuleList()
        for _ in range(n_ssm_layers):
            self.ssm_fwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_bwd.append(SelectiveSSM(d_model, d_state))
            self.ssm_merge.append(nn.Linear(2 * d_model, d_model))
            self.ssm_norm.append(nn.LayerNorm(d_model))
        self.ssm_drop = nn.Dropout(dropout)

        self.use_cross_attn = use_cross_attn
        if use_cross_attn:
            self.cross_attn = TemporalCrossAttention(d_model, n_heads=cross_attn_heads, dropout=dropout)

        self.prototypes = nn.Parameter(torch.randn(n_classes, d_model) * 0.02)
        self.proto_temp = nn.Parameter(torch.tensor(5.0))
        self.class_bias = nn.Parameter(torch.zeros(n_classes))
        self.fusion_alpha = nn.Parameter(torch.zeros(n_classes))

        self.n_families = 0
        self.family_head = None

    def init_prototypes_from_data(self, embeddings: torch.Tensor, labels: torch.Tensor) -> None:
        """Initializes class prototypes to the mean of their positive embeddings."""
        with torch.no_grad():
            h = self.input_proj(embeddings)
            for c in range(self.n_classes):
                mask = labels[:, c] > 0.5
                if mask.sum() > 0:
                    self.prototypes.data[c] = F.normalize(h[mask].mean(0), dim=0)

    def init_family_head(self, n_families: int, class_to_family: List[int]) -> None:
        """Initializes the auxiliary taxonomic classification head."""
        self.n_families = n_families
        self.family_head = nn.Linear(self.d_model, n_families)
        self.register_buffer('class_to_family', torch.tensor(class_to_family, dtype=torch.long))

    def forward(self, emb: torch.Tensor, perch_logits: Optional[torch.Tensor] = None, 
                site_ids: Optional[torch.Tensor] = None, hours: Optional[torch.Tensor] = None) -> Tuple[torch.Tensor, Optional[torch.Tensor], torch.Tensor]:
        B, T, _ = emb.shape

        h = self.input_proj(emb)
        h = h + self.pos_enc[:, :T, :]

        if site_ids is not None and hours is not None:
            s_emb = self.site_emb(site_ids)
            h_emb = self.hour_emb(hours)
            meta = self.meta_proj(torch.cat([s_emb, h_emb], dim=-1))
            h = h + meta[:, None, :]

        for fwd, bwd, merge, norm in zip(self.ssm_fwd, self.ssm_bwd, self.ssm_merge, self.ssm_norm):
            residual = h
            h_f = fwd(h)
            h_b = bwd(h.flip(1)).flip(1)
            h = merge(torch.cat([h_f, h_b], dim=-1))
            h = self.ssm_drop(h)
            h = norm(h + residual)

        if self.use_cross_attn:
            h = self.cross_attn(h)

        h_temporal = h

        h_norm = F.normalize(h, dim=-1)
        p_norm = F.normalize(self.prototypes, dim=-1)
        temp = F.softplus(self.proto_temp)
        sim = torch.matmul(h_norm, p_norm.T) * temp + self.class_bias[None, None, :]

        if perch_logits is not None:
            alpha = torch.sigmoid(self.fusion_alpha)[None, None, :]
            species_logits = alpha * sim + (1 - alpha) * perch_logits
        else:
            species_logits = sim

        family_logits = None
        if self.family_head is not None:
            h_pool = h.mean(dim=1)
            family_logits = self.family_head(h_pool)

        return species_logits, family_logits, h_temporal

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


class ResidualSSM(nn.Module):
    """
    Lightweight SSM trained strictly on the prediction residuals of the first-pass ensemble.
    Architecture: project(concat(emb, first_pass_scores)) -> 1-layer BiSSM -> linear correction head.
    """
    def __init__(self, d_input: int = 1536, d_scores: int = 234, d_model: int = 128, d_state: int = 16,
                 n_classes: int = 234, n_windows: int = 12, dropout: float = 0.1, n_sites: int = 20, meta_dim: int = 8):
        super().__init__()
        self.d_model = d_model
        
        self.input_proj = nn.Sequential(
            nn.Linear(d_input + d_scores, d_model),
            nn.LayerNorm(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        self.site_emb = nn.Embedding(n_sites, meta_dim)
        self.hour_emb = nn.Embedding(24, meta_dim)
        self.meta_proj = nn.Linear(2 * meta_dim, d_model)
        self.pos_enc = nn.Parameter(torch.randn(1, n_windows, d_model) * 0.02)

        self.ssm_fwd = SelectiveSSM(d_model, d_state)
        self.ssm_bwd = SelectiveSSM(d_model, d_state)
        self.ssm_merge = nn.Linear(2 * d_model, d_model)
        self.ssm_norm = nn.LayerNorm(d_model)
        self.ssm_drop = nn.Dropout(dropout)

        self.output_head = nn.Linear(d_model, n_classes)
        nn.init.zeros_(self.output_head.weight)
        nn.init.zeros_(self.output_head.bias)

    def forward(self, emb: torch.Tensor, first_pass_scores: torch.Tensor, 
                site_ids: Optional[torch.Tensor] = None, hours: Optional[torch.Tensor] = None) -> torch.Tensor:
        B, T, _ = emb.shape

        x = torch.cat([emb, first_pass_scores], dim=-1)
        h = self.input_proj(x)

        if site_ids is not None and hours is not None:
            site_e = self.site_emb(site_ids.clamp(0, self.site_emb.num_embeddings - 1))
            hour_e = self.hour_emb(hours.clamp(0, 23))
            meta = self.meta_proj(torch.cat([site_e, hour_e], dim=-1))
            h = h + meta.unsqueeze(1)

        h = h + self.pos_enc[:, :T, :]

        residual = h
        h_f = self.ssm_fwd(h)
        h_b = self.ssm_bwd(h.flip(1)).flip(1)
        h = self.ssm_merge(torch.cat([h_f, h_b], dim=-1))
        h = self.ssm_drop(h)
        h = self.ssm_norm(h + residual)

        correction = self.output_head(h)
        return correction

    def count_parameters(self) -> int:
        return sum(p.numel() for p in self.parameters() if p.requires_grad)

## 6. Training & Validation Engine

This modular engine manages the execution of single-fold training, Stochastic Weight Averaging (SWA), Knowledge Distillation, and OOF cross-validation. It dynamically handles the file-level batching and sequence augmentation.

In [7]:
class ProtoSSMTrainer:
    """Manages the training loop, SWA state, and validation for ProtoSSM."""
    
    def __init__(self, cfg: type(Config)):
        self.cfg = cfg
        self.train_cfg = cfg.PROTO_SSM_TRAIN

    def train_single_fold(self, model: nn.Module,
                          emb_train: np.ndarray, logits_train: np.ndarray, labels_train: np.ndarray,
                          site_ids_train: np.ndarray, hours_train: np.ndarray, file_families_train: Optional[np.ndarray],
                          emb_val: Optional[np.ndarray], logits_val: Optional[np.ndarray], labels_val: Optional[np.ndarray],
                          site_ids_val: Optional[np.ndarray], hours_val: Optional[np.ndarray], file_families_val: Optional[np.ndarray],
                          verbose: bool = True
                          ) -> Tuple[nn.Module, dict]:
        """Trains a single model instance using Focal Loss, Mixup, and SWA."""
        
        # Hyperparameters
        label_smoothing = self.train_cfg.get("label_smoothing", 0.0)
        mixup_alpha = self.train_cfg.get("mixup_alpha", 0.0)
        focal_gamma = self.train_cfg.get("focal_gamma", 0.0)
        n_epochs = self.train_cfg["n_epochs"]
        swa_start_epoch = int(n_epochs * self.train_cfg.get("swa_start_frac", 1.0))

        # Label Smoothing
        labels_np = labels_train.copy()
        if label_smoothing > 0:
            labels_np = labels_np * (1.0 - label_smoothing) + label_smoothing / 2.0

        # Guard for training without validation data (final fit on all data passes None).
        has_val = emb_val is not None
        if has_val:
            emb_v      = torch.tensor(emb_val,      dtype=torch.float32)
            logits_v   = torch.tensor(logits_val,   dtype=torch.float32)
            labels_v   = torch.tensor(labels_val,   dtype=torch.float32)
            site_v     = torch.tensor(site_ids_val, dtype=torch.long)
            hour_v     = torch.tensor(hours_val,    dtype=torch.long)
            fam_v      = torch.tensor(file_families_val, dtype=torch.float32) if file_families_val is not None else None

        # Class balancing
        labels_tr_t = torch.tensor(labels_np, dtype=torch.float32)
        pos_counts  = labels_tr_t.sum(dim=(0, 1))
        total       = labels_tr_t.shape[0] * labels_tr_t.shape[1]
        pos_weight  = ((total - pos_counts) / (pos_counts + 1)).clamp(max=self.train_cfg["pos_weight_cap"])

        optimizer = torch.optim.AdamW(model.parameters(), lr=self.train_cfg["lr"], weight_decay=self.train_cfg["weight_decay"])
        scheduler = TrainingUtils.get_scheduler(optimizer, n_epochs, 1, self.train_cfg["lr"])

        best_val_loss = float('inf')
        best_state    = None
        wait, swa_count = 0, 0
        swa_state = None
        history = {"train_loss": [], "val_loss": [], "val_auc": []}

        for epoch in range(n_epochs):
            # Dynamic Mixup (skip first 5 epochs as warm-up)
            if mixup_alpha > 0 and epoch > 5:  # skip mixup for first 5 warmup epochs
                emb_mix, logits_mix, labels_mix, _, _, fam_mix = TrainingUtils.mixup_files(
                    emb_train, logits_train, labels_np,
                    site_ids_train, hours_train, file_families_train,
                    alpha=mixup_alpha,
                )
            else:
                emb_mix, logits_mix, labels_mix = emb_train, logits_train, labels_np
                fam_mix = file_families_train

            emb_mix    = torch.tensor(emb_mix,    dtype=torch.float32)
            logits_mix = torch.tensor(logits_mix, dtype=torch.float32)
            labels_mix = torch.tensor(labels_mix, dtype=torch.float32)
            site_tr    = torch.tensor(site_ids_train, dtype=torch.long)
            hour_tr    = torch.tensor(hours_train,    dtype=torch.long)
            fam_tr     = torch.tensor(fam_mix, dtype=torch.float32) if fam_mix is not None else None

            # Training Pass
            model.train()
            species_out, family_out, _ = model(emb_mix, logits_mix, site_ids=site_tr, hours=hour_tr)

            # Loss Computation
            if focal_gamma > 0:
                loss_main = TrainingUtils.focal_bce_with_logits(species_out, labels_mix, gamma=focal_gamma, pos_weight=pos_weight[None, None, :])
            else:
                loss_main = F.binary_cross_entropy_with_logits(species_out, labels_mix, pos_weight=pos_weight[None, None, :])

            loss_distill = F.mse_loss(species_out, logits_mix)
            loss = loss_main + self.train_cfg["distill_weight"] * loss_distill

            if family_out is not None and fam_tr is not None:
                loss += 0.1 * F.binary_cross_entropy_with_logits(family_out, fam_tr)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step()

            # SWA Accumulation
            if epoch >= swa_start_epoch:
                if swa_state is None:
                    swa_state = {k: v.clone() for k, v in model.state_dict().items()}
                    swa_count = 1
                else:
                    for k in swa_state:
                        swa_state[k] += model.state_dict()[k]
                    swa_count += 1

            # Conditional validation pass - when training on all data (no val set),
            # fall back to tracking train loss for early stopping.
            model.eval()
            with torch.no_grad():
                if has_val:
                    val_out, _, _ = model(emb_v, logits_v, site_ids=site_v, hours=hour_v)
                    val_loss = F.binary_cross_entropy_with_logits(val_out, labels_v, pos_weight=pos_weight[None, None, :])
                    val_pred = val_out.reshape(-1, val_out.shape[-1]).numpy()
                    val_true = labels_v.reshape(-1, labels_v.shape[-1]).numpy()
                    try:
                        keep    = val_true.sum(axis=0) > 0
                        val_auc = roc_auc_score(val_true[:, keep], val_pred[:, keep], average="macro")
                    except Exception:
                        val_auc = 0.0
                else:
                    val_loss = loss   # track training loss when no validation set
                    val_auc  = 0.0

            history["train_loss"].append(loss.item())
            history["val_loss"].append(val_loss.item())
            history["val_auc"].append(val_auc)

            # Checkpointing & Early Stopping
            if val_loss.item() < best_val_loss:
                best_val_loss = val_loss.item()
                best_state    = {k: v.clone() for k, v in model.state_dict().items()}
                wait = 0
            else:
                wait += 1

            if verbose and (epoch + 1) % 20 == 0:
                lr_now   = optimizer.param_groups[0]['lr']
                swa_info = f" swa={swa_count}" if swa_count > 0 else ""
                print(f"  Epoch {epoch+1:3d}: train={loss.item():.4f} val={val_loss.item():.4f} "
                      f"auc={val_auc:.4f} lr={lr_now:.6f} wait={wait}{swa_info}")

            if wait >= self.train_cfg["patience"]:
                break

        # Post-training State Resolution
        if swa_state is not None and swa_count >= 3:
            if verbose:
                print(f"  Applying SWA (averaged {swa_count} checkpoints)")
            avg_state = {k: v / swa_count for k, v in swa_state.items()}
            model.load_state_dict(avg_state)
        elif best_state is not None:
            model.load_state_dict(best_state)

        if verbose:
            print(f"  Training complete. Best val_loss={best_val_loss:.4f}")
            with torch.no_grad():
                alphas = torch.sigmoid(model.fusion_alpha).numpy()
                print(f"  Fusion alpha: mean={alphas.mean():.3f} min={alphas.min():.3f} max={alphas.max():.3f}")
                print(f"  Proto temperature: {F.softplus(model.proto_temp).item():.3f}")

        return model, history

## 7. Acoustic Feature Extraction & Caching

This module wraps the Google Perch base model. Since Perch inference is computationally expensive on the CPU, we aggressively cache the full-file embeddings and initial logits. The engine handles audio resampling, batching, and extracting multi-window representations.

In [8]:
class AudioFeatureExtractor:
    def __init__(self, cfg: type(Config), tax_manager: TaxonomyManager, perch_mappings: dict):
        self.cfg = cfg
        self.tax_manager = tax_manager
        self.mappings = perch_mappings
        self.birdclassifier = tf.saved_model.load(str(self.cfg.MODEL_DIR))
        self.infer_fn = self.birdclassifier.signatures["serving_default"]

    def infer_perch_with_embeddings(self, paths: List[Path], proxy_reduce: str = "max") -> Tuple[pd.DataFrame, np.ndarray, np.ndarray]:
        n_rows = len(paths) * self.cfg.N_WINDOWS
        row_ids = np.empty(n_rows, dtype=object)
        filenames = np.empty(n_rows, dtype=object)
        sites, hours = np.empty(n_rows, dtype=object), np.empty(n_rows, dtype=np.int16)

        scores = np.zeros((n_rows, self.tax_manager.n_classes), dtype=np.float32)
        embeddings = np.zeros((n_rows, 1536), dtype=np.float32)

        write_row = 0
        mapped_pos = self.mappings["mapped_pos"]
        mapped_bc_indices = self.mappings["bc_indices"][self.mappings["mapped_mask"]].astype(np.int32)

        for start in range(0, len(paths), self.cfg.BATCH_FILES):
            batch_paths = paths[start:start + self.cfg.BATCH_FILES]
            x = np.empty((len(batch_paths) * self.cfg.N_WINDOWS, self.cfg.WINDOW_SAMPLES), dtype=np.float32)
            batch_row_start = write_row
            x_pos = 0

            for path in batch_paths:
                y, sr = sf.read(path, dtype="float32", always_2d=False)
                y = y.mean(axis=1) if y.ndim == 2 else y
                y = np.pad(y, (0, max(0, self.cfg.FILE_SAMPLES - len(y))))[:self.cfg.FILE_SAMPLES]
                x[x_pos:x_pos + self.cfg.N_WINDOWS] = y.reshape(self.cfg.N_WINDOWS, self.cfg.WINDOW_SAMPLES)

                m = re.match(r"BC2026_(?:Train|Test)_(\d+)_(S\d+)_(\d{8})_(\d{6})\.ogg", path.name)
                row_ids[write_row:write_row + self.cfg.N_WINDOWS] = [f"{path.stem}_{t}" for t in range(5, 65, 5)]
                filenames[write_row:write_row + self.cfg.N_WINDOWS] = path.name
                sites[write_row:write_row + self.cfg.N_WINDOWS] = m.group(2) if m else None
                hours[write_row:write_row + self.cfg.N_WINDOWS] = int(m.group(4)[:2]) if m else -1

                x_pos += self.cfg.N_WINDOWS
                write_row += self.cfg.N_WINDOWS

            outputs = self.infer_fn(inputs=tf.convert_to_tensor(x))
            logits = outputs["label"].numpy().astype(np.float32, copy=False)
            emb = outputs["embedding"].numpy().astype(np.float32, copy=False)

            scores[batch_row_start:write_row, mapped_pos] = logits[:, mapped_bc_indices]
            embeddings[batch_row_start:write_row] = emb
            
            # Map Proxy Scores
            for pos, bc_idx_arr in self.tax_manager.selected_proxy_pos_to_bc.items():
                sub = logits[:, bc_idx_arr]
                proxy_score = sub.max(axis=1) if proxy_reduce == "max" else sub.mean(axis=1)
                scores[batch_row_start:write_row, pos] = proxy_score.astype(np.float32)

            del x, outputs, logits, emb
            gc.collect()

        meta_df = pd.DataFrame({"row_id": row_ids, "filename": filenames, "site": sites, "hour_utc": hours})
        return meta_df, scores, embeddings

## 8. Prior Metadata Engine & Probe Features

Species distributions vary drastically by location and time of day. This module creates `site x hour` prior tables. We also engineer temporal and structural features (e.g., standard deviation across the file, diff from file mean to detect onset/offset) for the MLP Probes.

In [9]:
class PriorAndProbeManager:
    @staticmethod
    def fit_prior_tables(prior_df: pd.DataFrame, y_prior: np.ndarray) -> dict:
        # Reset the index so Pandas groupby indices perfectly match the NumPy array
        prior_df = prior_df.reset_index(drop=True)
        
        global_p = y_prior.mean(axis=0).astype(np.float32)
        tables = {"global_p": global_p}
        
        for col, key_p, key_n, key_i in [("site", "site_p", "site_n", "site_to_i"), ("hour_utc", "hour_p", "hour_n", "hour_to_i")]:
            keys = sorted(prior_df[col].dropna().unique().tolist())
            tables[key_i] = {k: i for i, k in enumerate(keys)}
            tables[key_n] = np.zeros(len(keys), dtype=np.float32)
            tables[key_p] = np.zeros((len(keys), y_prior.shape[1]), dtype=np.float32)
            for k in keys:
                mask = prior_df[col].values == k
                tables[key_n][tables[key_i][k]] = mask.sum()
                tables[key_p][tables[key_i][k]] = y_prior[mask].mean(axis=0)

        # Restore Joint Site-Hour Prior
        sh_to_i, sh_n_list, sh_p_list = {}, [], []
        for (s, h), idx in prior_df.groupby(["site", "hour_utc"]).groups.items():
            sh_to_i[(str(s), int(h))] = len(sh_n_list)
            sh_n_list.append(len(idx))
            sh_p_list.append(y_prior[np.array(list(idx))].mean(axis=0))
            
        tables["sh_to_i"] = sh_to_i
        tables["sh_n"] = np.array(sh_n_list, dtype=np.float32)
        tables["sh_p"] = np.stack(sh_p_list).astype(np.float32) if sh_p_list else np.zeros((0, y_prior.shape[1]), dtype=np.float32)
        return tables

    @staticmethod
    def prior_logits_from_tables(sites: np.ndarray, hours: np.ndarray, tables: dict, eps: float = 1e-4) -> np.ndarray:
        n = len(sites)
        p = np.repeat(tables["global_p"][None, :], n, axis=0).astype(np.float32)
        
        hour_idx = np.array([tables["hour_to_i"].get(int(h), -1) for h in hours])
        site_idx = np.array([tables["site_to_i"].get(str(s), -1) for s in sites])
        sh_idx = np.array([tables["sh_to_i"].get((str(s), int(h)), -1) for s, h in zip(sites, hours)])

        for idx, key_n, key_p, weight_factor in [(hour_idx, "hour_n", "hour_p", 8.0), (site_idx, "site_n", "site_p", 8.0), (sh_idx, "sh_n", "sh_p", 4.0)]:
            valid = idx >= 0
            if valid.any():
                n_val = tables[key_n][idx[valid]][:, None]
                w = n_val / (n_val + weight_factor)
                p[valid] = w * tables[key_p][idx[valid]] + (1.0 - w) * p[valid]
                
        np.clip(p, eps, 1.0 - eps, out=p)
        return (np.log(p) - np.log1p(-p)).astype(np.float32)

    @staticmethod
    def smooth_cols(scores: np.ndarray, cols: np.ndarray, alpha: float, mode: str, n_windows: int = 12) -> np.ndarray:
        if alpha <= 0 or len(cols) == 0: return scores.copy()
        s = scores.copy()
        view = s.reshape(-1, n_windows, s.shape[1])
        x = view[:, :, cols]
        prev_x, next_x = np.concatenate([x[:, :1, :], x[:, :-1, :]], axis=1), np.concatenate([x[:, 1:, :], x[:, -1:, :]], axis=1)
        if mode == "texture":
            view[:, :, cols] = (1.0 - alpha) * x + 0.5 * alpha * (prev_x + next_x)
        else:
            view[:, :, cols] = (1.0 - alpha) * x + alpha * np.maximum(x, np.maximum(prev_x, next_x))
        return s

    @staticmethod
    def fuse_scores_with_tables(base_scores: np.ndarray, sites: np.ndarray, hours: np.ndarray, tables: dict, tax_mgr: TaxonomyManager) -> Tuple[np.ndarray, np.ndarray]:
        """Applies mathematical fusion of Prior logits with raw Perch logits."""
        scores = base_scores.copy()
        prior = PriorAndProbeManager.prior_logits_from_tables(sites, hours, tables)
        idx, best = tax_mgr.indices, Config.BEST_FUSION
        
        if len(idx["mapped_active_event"]): scores[:, idx["mapped_active_event"]] += best["lambda_event"] * prior[:, idx["mapped_active_event"]]
        if len(idx["mapped_active_texture"]): scores[:, idx["mapped_active_texture"]] += best["lambda_texture"] * prior[:, idx["mapped_active_texture"]]
        if len(idx["selected_proxy_active_texture"]): scores[:, idx["selected_proxy_active_texture"]] += best["lambda_proxy_texture"] * prior[:, idx["selected_proxy_active_texture"]]
        if len(idx["selected_prioronly_active_event"]): scores[:, idx["selected_prioronly_active_event"]] = best["lambda_event"] * prior[:, idx["selected_prioronly_active_event"]]
        if len(idx["selected_prioronly_active_texture"]): scores[:, idx["selected_prioronly_active_texture"]] = best["lambda_texture"] * prior[:, idx["selected_prioronly_active_texture"]]
        if len(idx["unmapped_inactive"]): scores[:, idx["unmapped_inactive"]] = -8.0

        scores = PriorAndProbeManager.smooth_cols(scores, idx["active_texture"], best["smooth_texture"], "texture")
        scores = PriorAndProbeManager.smooth_cols(scores, idx["active_event"], best["smooth_event"], "event")
        return scores.astype(np.float32), prior

    @staticmethod
    def build_oof_base_prior(scores_raw, meta_df, tax_mgr) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
        """Leak-free OOF Base and Prior features."""
        oof_base, oof_prior, fold_id = np.zeros_like(scores_raw), np.zeros_like(scores_raw), np.full(len(meta_df), -1, dtype=np.int16)
        for fold, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=5).split(scores_raw, groups=meta_df["filename"].values), 1):
            val_files = set(meta_df.iloc[va_idx]["filename"])
            prior_mask = ~tax_mgr.sc_clean["filename"].isin(val_files).values
            tables = PriorAndProbeManager.fit_prior_tables(tax_mgr.sc_clean.loc[prior_mask], tax_mgr.y_sc[prior_mask])
            
            va_base, va_prior = PriorAndProbeManager.fuse_scores_with_tables(scores_raw[va_idx], meta_df.iloc[va_idx]["site"].values, meta_df.iloc[va_idx]["hour_utc"].values, tables, tax_mgr)
            oof_base[va_idx], oof_prior[va_idx], fold_id[va_idx] = va_base, va_prior, fold
        return oof_base, oof_prior, fold_id
    
    @staticmethod
    def build_class_features(emb_proj, raw_col, prior_col, base_col):
        prev_base, next_base = np.concatenate([base_col.reshape(-1, 12)[:, :1], base_col.reshape(-1, 12)[:, :-1]], axis=1).reshape(-1), np.concatenate([base_col.reshape(-1, 12)[:, 1:], base_col.reshape(-1, 12)[:, -1:]], axis=1).reshape(-1)
        mean_base = np.repeat(base_col.reshape(-1, 12).mean(axis=1), 12)
        return np.column_stack([emb_proj, raw_col, prior_col, base_col, prev_base, next_base, mean_base, np.repeat(base_col.reshape(-1, 12).max(axis=1), 12), np.repeat(base_col.reshape(-1, 12).std(axis=1), 12), base_col - mean_base, base_col - prev_base, base_col - next_base, raw_col * prior_col, raw_col * base_col, prior_col * base_col]).astype(np.float32)

## 9. Main Pipeline Execution

This script orchestrates the entire workflow. It detects whether we are in `train` or `submit` mode.

In [10]:
def run_pipeline():
    cfg = Config()

    # 1. Setup 
    tax_manager = TaxonomyManager(cfg)
    tax_manager.build_label_matrices()
    tax_manager.build_taxonomy_groups()
    perch_mappings = tax_manager.map_perch_indices()
    extractor = AudioFeatureExtractor(cfg, tax_manager, perch_mappings)

    # Load training cache
    meta_full_path = cfg.FULL_CACHE_INPUT_DIR / "full_perch_meta.parquet"
    npz_full_path  = cfg.FULL_CACHE_INPUT_DIR / "full_perch_arrays.npz"
    if not (meta_full_path.exists() and npz_full_path.exists()):
        print("Cache not found — exiting.")
        return

    meta_full = pd.read_parquet(meta_full_path)
    arr = np.load(npz_full_path)
    scores_full_raw = arr["scores_full_raw"].astype(np.float32)
    emb_full        = arr["emb_full"].astype(np.float32)

    # Align ground-truth labels to cached file order (must match meta_full row_id order)
    full_truth = (tax_manager.sc_clean[tax_manager.sc_clean["file_fully_labeled"]]
                  .sort_values(["filename", "end_sec"]).reset_index(drop=False))
    full_truth_aligned = full_truth.set_index("row_id").loc[meta_full["row_id"]].reset_index()
    Y_FULL = tax_manager.y_sc[full_truth_aligned["index"].to_numpy()]

    # 2. OOF Base/Prior + MLP Probes 
    print("Building OOF Features and Training MLP Probes...")
    oof_base, oof_prior, _ = PriorAndProbeManager.build_oof_base_prior(
        scores_full_raw, meta_full, tax_manager
    )

    emb_scaler = StandardScaler()
    emb_full_scaled = emb_scaler.fit_transform(emb_full)
    n_comp  = min(int(cfg.FROZEN_BEST_PROBE["pca_dim"]),
                  emb_full_scaled.shape[0] - 1, emb_full_scaled.shape[1])
    emb_pca = PCA(n_components=n_comp)
    Z_FULL  = emb_pca.fit_transform(emb_full_scaled).astype(np.float32)

    probe_models = {}
    valid_classes = np.where(Y_FULL.sum(axis=0) >= cfg.FROZEN_BEST_PROBE["min_pos"])[0]
    for cls_idx in tqdm(valid_classes, desc="Training MLP probes", disable=not cfg.VERBOSE):
        y = Y_FULL[:, cls_idx]
        if y.sum() == 0 or y.sum() == len(y):
            continue
        X_cls = PriorAndProbeManager.build_class_features(
            Z_FULL, scores_full_raw[:, cls_idx],
            oof_prior[:, cls_idx], oof_base[:, cls_idx]
        )
        # Positive oversampling for rare species.
        # Without this, MLPClassifier sees severely imbalanced classes and ignores the
        # positive signal for rare birds - exactly the species that matter most.
        n_pos, n_neg = int(y.sum()), len(y) - int(y.sum())
        if n_pos > 0 and n_neg > n_pos:
            repeat  = max(1, n_neg // n_pos)
            pos_idx = np.where(y == 1)[0]
            X_bal   = np.vstack([X_cls, np.tile(X_cls[pos_idx], (repeat, 1))])
            y_bal   = np.concatenate([y, np.ones(len(pos_idx) * repeat, dtype=y.dtype)])
        else:
            X_bal, y_bal = X_cls, y
        clf = MLPClassifier(**cfg.MLP_PARAMS)
        clf.fit(X_bal, y_bal)
        probe_models[cls_idx] = clf
    print(f"MLP probes trained: {len(probe_models)}")

    ENSEMBLE_WEIGHT_PROTO = 0.50
    print(f"\nUsing default ensemble weight: ProtoSSM={ENSEMBLE_WEIGHT_PROTO:.2f}")

    # 3. Train ProtoSSM v5 
    print("Training ProtoSSM v5...")

    # reshape_to_files: (n_rows, feat) -> (n_files, 12, feat)
    # Using dict.fromkeys preserves insertion order and deduplicates, giving the
    # same file ordering as the flat array (critical for correct label alignment).
    def reshape_to_files(flat_arr, meta):
        unique_files = list(dict.fromkeys(meta["filename"].tolist()))
        return flat_arr.reshape(len(unique_files), cfg.N_WINDOWS, -1), unique_files

    emb_files,    file_list = reshape_to_files(emb_full,        meta_full)
    logits_files, _         = reshape_to_files(scores_full_raw, meta_full)
    labels_files, _         = reshape_to_files(Y_FULL,          meta_full)

    site_to_idx = {s: i + 1 for i, s in enumerate(meta_full["site"].unique())}
    first_meta  = meta_full.groupby("filename").first()
    site_ids_all = np.array([min(site_to_idx.get(s, 0), cfg.PROTO_SSM["n_sites"] - 1)
                              for s in first_meta["site"]], dtype=np.int64)
    hours_all    = np.array([int(h) % 24 for h in first_meta["hour_utc"]], dtype=np.int64)

    # Per-file multi-hot family labels for the auxiliary taxonomic loss
    file_families = np.zeros((len(file_list), tax_manager.n_families), dtype=np.float32)
    for fi in range(len(file_list)):
        for ci in np.where(labels_files[fi].sum(axis=0) > 0)[0]:
            file_families[fi, tax_manager.class_to_family[ci]] = 1.0

    t0_proto = time.time()

    # Pass n_classes dynamically from tax_manager instead of relying on the
    # hardcoded default of 234 in ProtoSSMv5.__init__.  If the competition label
    # count ever differs, the default silently builds a model with the wrong output dim.
    ssm_kwargs = {k: v for k, v in cfg.PROTO_SSM.items() if k != "n_prototypes"}
    model = ProtoSSMv5(n_classes=tax_manager.n_classes, **ssm_kwargs).to(cfg.DEVICE)
    model.init_prototypes_from_data(
        torch.tensor(emb_full,  dtype=torch.float32),
        torch.tensor(Y_FULL,    dtype=torch.float32)
    )
    model.init_family_head(tax_manager.n_families, tax_manager.class_to_family)

    trainer = ProtoSSMTrainer(cfg)
    # Train on ALL data (no validation split for the final model).
    # train_single_fold now handles None val arguments gracefully.
    model, train_history = trainer.train_single_fold(
        model, emb_files, logits_files, labels_files.astype(np.float32),
        site_ids_all, hours_all, file_families,
        None, None, None, None, None, None,
        verbose=True,  # always verbose for the final model, matching Reference
    )
    train_time = time.time() - t0_proto
    print(f"Final model training time: {train_time:.1f}s")

    with torch.no_grad():
        final_alphas = torch.sigmoid(model.fusion_alpha).numpy()
        print(f"Fusion alpha: mean={final_alphas.mean():.4f} min={final_alphas.min():.4f} max={final_alphas.max():.4f}")

    # 4. Train ResidualSSM 
    print("Training Residual SSM...")

    final_prior_tables = PriorAndProbeManager.fit_prior_tables(
        tax_manager.sc_clean.reset_index(drop=True), tax_manager.y_sc
    )
    train_base_scores, train_prior_scores = PriorAndProbeManager.fuse_scores_with_tables(
        scores_full_raw, meta_full["site"].values, meta_full["hour_utc"].values,
        final_prior_tables, tax_manager
    )
    mlp_train_flat = train_base_scores.copy()
    alpha_p = float(cfg.FROZEN_BEST_PROBE["alpha"])
    for cls_idx, clf in probe_models.items():
        X_cls = PriorAndProbeManager.build_class_features(
            Z_FULL, scores_full_raw[:, cls_idx],
            train_prior_scores[:, cls_idx], train_base_scores[:, cls_idx]
        )
        prob = clf.predict_proba(X_cls)[:, 1].astype(np.float32)
        pred = np.log(prob + 1e-7) - np.log(1 - prob + 1e-7)
        mlp_train_flat[:, cls_idx] = (1.0 - alpha_p) * train_base_scores[:, cls_idx] + alpha_p * pred
    mlp_train_files, _ = reshape_to_files(mlp_train_flat, meta_full)

    # ProtoSSM first-pass on training data
    model.eval()
    with torch.no_grad():
        proto_out, _, _ = model(
            torch.tensor(emb_files,    dtype=torch.float32),
            torch.tensor(logits_files, dtype=torch.float32),
            site_ids=torch.tensor(site_ids_all, dtype=torch.long),
            hours   =torch.tensor(hours_all,    dtype=torch.long)
        )
    first_pass_files = (
        ENSEMBLE_WEIGHT_PROTO * proto_out.numpy() +
        (1.0 - ENSEMBLE_WEIGHT_PROTO) * mlp_train_files
    ).astype(np.float32)

    # Residual = ground-truth − sigmoid(first_pass)  ∈ [−1, 1]
    residuals = labels_files.astype(np.float32) - (1.0 / (1.0 + np.exp(-first_pass_files)))
    print(f"Residuals: mean={residuals.mean():.4f}, abs_mean={np.abs(residuals).mean():.4f}")

    # Instantiate ResidualSSM with explicit constructor arguments.
    res_cfg   = cfg.RESIDUAL_SSM
    res_model = ResidualSSM(
        d_input  =emb_full.shape[1],
        d_scores =tax_manager.n_classes,
        d_model  =res_cfg["d_model"],
        d_state  =res_cfg["d_state"],
        n_classes=tax_manager.n_classes,
        n_windows=cfg.N_WINDOWS,
        dropout  =res_cfg["dropout"],
        n_sites  =cfg.PROTO_SSM["n_sites"],
        meta_dim =8,
    ).to(cfg.DEVICE)

    # Full training loop with 85/15 train/val split, OneCycleLR, grad clipping,
    # and patience-based early stopping.
    n_files = len(file_list)
    n_val   = max(1, int(n_files * 0.15))
    perm    = torch.randperm(n_files, generator=torch.Generator().manual_seed(123))
    val_i, train_i = perm[:n_val].numpy(), perm[n_val:].numpy()

    emb_tr_t  = torch.tensor(emb_files[train_i],        dtype=torch.float32)
    fp_tr_t   = torch.tensor(first_pass_files[train_i], dtype=torch.float32)
    res_tr_t  = torch.tensor(residuals[train_i],         dtype=torch.float32)
    site_tr_t = torch.tensor(site_ids_all[train_i],      dtype=torch.long)
    hour_tr_t = torch.tensor(hours_all[train_i],          dtype=torch.long)
    emb_va_t  = torch.tensor(emb_files[val_i],           dtype=torch.float32)
    fp_va_t   = torch.tensor(first_pass_files[val_i],   dtype=torch.float32)
    res_va_t  = torch.tensor(residuals[val_i],            dtype=torch.float32)
    site_va_t = torch.tensor(site_ids_all[val_i],         dtype=torch.long)
    hour_va_t = torch.tensor(hours_all[val_i],             dtype=torch.long)

    res_opt   = torch.optim.AdamW(res_model.parameters(), lr=res_cfg["lr"], weight_decay=1e-3)
    res_sched = TrainingUtils.get_scheduler(res_opt, res_cfg["n_epochs"], 1, res_cfg["lr"])
    best_res_loss, best_res_state, res_wait = float('inf'), None, 0

    for epoch in range(res_cfg["n_epochs"]):
        res_model.train()
        correction = res_model(emb_tr_t, fp_tr_t, site_ids=site_tr_t, hours=hour_tr_t)
        loss = F.mse_loss(correction, res_tr_t)
        res_opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(res_model.parameters(), 1.0)
        res_opt.step()
        res_sched.step()

        res_model.eval()
        with torch.no_grad():
            val_corr = res_model(emb_va_t, fp_va_t, site_ids=site_va_t, hours=hour_va_t)
            val_loss = F.mse_loss(val_corr, res_va_t)

        if val_loss.item() < best_res_loss:
            best_res_loss  = val_loss.item()
            best_res_state = {k: v.clone() for k, v in res_model.state_dict().items()}
            res_wait = 0
        else:
            res_wait += 1
        if (epoch + 1) % 20 == 0:
            print(f"  ResidualSSM epoch {epoch+1}: train={loss.item():.6f} val={val_loss.item():.6f} wait={res_wait}")
        if res_wait >= res_cfg["patience"]:
            print(f"  ResidualSSM early stop at epoch {epoch+1}")
            break
    if best_res_state is not None:
        res_model.load_state_dict(best_res_state)
    print(f"ResidualSSM training complete. Best val MSE: {best_res_loss:.6f}")
    CORRECTION_WEIGHT = res_cfg["correction_weight"]

    # 5. Test Inference 
    print("Running Test Inference...")

    # Dry-run fallback uses train_soundscapes.
    test_paths = sorted((cfg.BASE_DIR / "test_soundscapes").glob("*.ogg"))
    if not test_paths:
        print(f"No hidden test files. Dry-run on first {cfg.DRYRUN_N_FILES} train soundscapes.")
        test_paths = sorted((cfg.BASE_DIR / "train_soundscapes").glob("*.ogg"))[:cfg.DRYRUN_N_FILES]
    else:
        print(f"Hidden test files: {len(test_paths)}")

    meta_test, scores_test_raw, emb_test = extractor.infer_perch_with_embeddings(
        test_paths, proxy_reduce=cfg.PROXY_REDUCE
    )

    # Prior fusion on test data (uses final tables fitted on all training data)
    test_base_scores, test_prior_scores = PriorAndProbeManager.fuse_scores_with_tables(
        scores_test_raw, meta_test["site"].values, meta_test["hour_utc"].values,
        final_prior_tables, tax_manager
    )

    # MLP probe predictions on test data
    Z_TEST     = emb_pca.transform(emb_scaler.transform(emb_test)).astype(np.float32)
    mlp_scores = test_base_scores.copy()
    for cls_idx, clf in probe_models.items():
        pred = clf.predict_proba(PriorAndProbeManager.build_class_features(
            Z_TEST, scores_test_raw[:, cls_idx],
            test_prior_scores[:, cls_idx], test_base_scores[:, cls_idx]
        ))[:, 1].astype(np.float32)
        mlp_scores[:, cls_idx] = (
            (1.0 - alpha_p) * test_base_scores[:, cls_idx] +
            alpha_p * (np.log(pred + 1e-7) - np.log(1 - pred + 1e-7))
        )

    emb_test_files,    test_file_list = reshape_to_files(emb_test,        meta_test)
    logits_test_files, _              = reshape_to_files(scores_test_raw, meta_test)
    test_first_meta = meta_test.groupby("filename").first()
    test_site_ids   = np.array([min(site_to_idx.get(s, 0), cfg.PROTO_SSM["n_sites"] - 1)
                                 for s in test_first_meta["site"]], dtype=np.int64)
    test_hours      = np.array([int(h) % 24 for h in test_first_meta["hour_utc"]], dtype=np.int64)

    # ProtoSSM inference with TTA (temporal shift augmentation)
    model.eval()
    all_preds = []
    for shift in cfg.TTA_SHIFTS:
        e = emb_test_files    if shift == 0 else np.roll(emb_test_files,    shift, axis=1)
        l = logits_test_files if shift == 0 else np.roll(logits_test_files, shift, axis=1)
        with torch.no_grad():
            out, _, _ = model(
                torch.tensor(e, dtype=torch.float32),
                torch.tensor(l, dtype=torch.float32),
                site_ids=torch.tensor(test_site_ids, dtype=torch.long),
                hours   =torch.tensor(test_hours,    dtype=torch.long)
            )
            pred = out.numpy()
        if shift != 0:
            pred = np.roll(pred, -shift, axis=1)
        all_preds.append(pred)
    proto_scores_flat = np.mean(all_preds, axis=0).reshape(-1, tax_manager.n_classes).astype(np.float32)

    # Weighted ensemble of ProtoSSM + MLP probes
    final_test_scores = (
        ENSEMBLE_WEIGHT_PROTO * proto_scores_flat +
        (1.0 - ENSEMBLE_WEIGHT_PROTO) * mlp_scores
    ).astype(np.float32)

    # Residual SSM second-pass correction
    first_pass_test_files, _ = reshape_to_files(final_test_scores, meta_test)
    res_model.eval()
    with torch.no_grad():
        test_correction = res_model(
            torch.tensor(emb_test_files,        dtype=torch.float32),
            torch.tensor(first_pass_test_files, dtype=torch.float32),
            site_ids=torch.tensor(test_site_ids, dtype=torch.long),
            hours   =torch.tensor(test_hours,    dtype=torch.long)
        ).numpy()
    final_test_scores = final_test_scores + CORRECTION_WEIGHT * test_correction.reshape(-1, tax_manager.n_classes)
    print(f"Residual correction applied (weight={CORRECTION_WEIGHT}): "
          f"abs_mean={np.abs(test_correction).mean():.4f}")

    # 6. Post-Processing 
    class_temperatures = np.ones(tax_manager.n_classes, dtype=np.float32) * cfg.TEMPERATURE["aves"]
    for ci, label in enumerate(tax_manager.primary_labels):
        if perch_mappings["class_name_map"].get(label, "Aves") in {"Amphibia", "Insecta"}:
            class_temperatures[ci] = cfg.TEMPERATURE["texture"]

    # Temperature scaling -> sigmoid -> progressive post-processing chain
    probs = 1.0 / (1.0 + np.exp(-np.clip(final_test_scores / class_temperatures[None, :], -30, 30)))

    if cfg.FILE_LEVEL_TOP_K > 0:
        probs = PostProcessor.file_level_confidence_scale(probs, cfg.N_WINDOWS, cfg.FILE_LEVEL_TOP_K)
        probs = np.clip(probs, 0.0, 1.0)
    if cfg.RANK_AWARE_SCALE:
        probs = PostProcessor.rank_aware_scaling(probs, cfg.N_WINDOWS, cfg.RANK_AWARE_POWER)
        probs = np.clip(probs, 0.0, 1.0)
    if cfg.DELTA_SHIFT_ALPHA > 0:
        probs = PostProcessor.adaptive_delta_smooth(probs, cfg.N_WINDOWS, cfg.DELTA_SHIFT_ALPHA)
        probs = np.clip(probs, 0.0, 1.0)

    per_class_thresholds = np.full(tax_manager.n_classes, 0.5, dtype=np.float32)
    probs = PostProcessor.apply_per_class_thresholds(probs, per_class_thresholds)
    probs = np.clip(probs, 0.0, 1.0)

    # 7. Build Submission 
    submission = pd.DataFrame(probs, columns=tax_manager.primary_labels)
    submission.insert(0, "row_id", meta_test["row_id"].values)
    submission.to_csv("submission.csv", index=False)
    print(f"Complete. Saved submission.csv  shape={submission.shape}")
    print(f"Mean probability: {probs.mean():.4f} | range [{probs.min():.6f}, {probs.max():.6f}]")

if __name__ == "__main__":
    run_pipeline()

2026-04-04 10:56:31.155948: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Building OOF Features and Training MLP Probes...
MLP probes trained: 58

Using default ensemble weight: ProtoSSM=0.50
Training ProtoSSM v5...
  Epoch  20: train=0.5010 val=0.5010 auc=0.0000 lr=0.000737 wait=0
  Epoch  40: train=0.4941 val=0.4941 auc=0.0000 lr=0.000452 wait=2
  Applying SWA (averaged 6 checkpoints)
  Training complete. Best val_loss=0.4855
  Fusion alpha: mean=0.501 min=0.494 max=0.506
  Proto temperature: 5.029
Final model training time: 65.2s
Fusion alpha: mean=0.5008 min=0.4936 max=0.5058
Training Residual SSM...
Residuals: mean=-0.4290, abs_mean=0.4379
  ResidualSSM epoch 20: train=0.019062 val=0.020283 wait=0
  ResidualSSM epoch 40: train=0.016888 val=0.018610 wait=10
ResidualSSM training complete. Best val MSE: 0.018457
Running Test Inference...
No hidden test files. Dry-run on first 20 train soundscapes.


I0000 00:00:1775300311.301029      69 device_compiler.h:196] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


Residual correction applied (weight=0.35): abs_mean=0.4606
Complete. Saved submission.csv  shape=(240, 235)
Mean probability: 0.2445 | range [0.000000, 0.999822]
